In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import openpyxl
from pandas.api.types import CategoricalDtype
import warnings
warnings.filterwarnings("ignore")

# STAT390 Legal Aid Project, Sarah Abara
*Cleaning and Processing Data for Final Tableau Dashboard*

In [ ]:
# List of (month label, file path) tuples
month_files = [
    ('April 2024', 'All Calls by Month/April 2024.xlsx'),
    ('August 2024', 'All Calls by Month/August 2024.xlsx'),
    ('December 2024', 'All Calls by Month/December 2024.xlsx'),
    ('February 2025', 'All Calls by Month/February 2025.xlsx'),
    ('January 2025', 'All Calls by Month/January 2025.xlsx'),
    ('July 2024', 'All Calls by Month/July 2024.xlsx'),
    ('June 2024', 'All Calls by Month/June 2024.xlsx'),
    ('March 2025', 'All Calls by Month/March 2025.xlsx'),
    ('May 2024', 'All Calls by Month/May 2024.xlsx'),
    ('October 2024', 'All Calls by Month/October 2024.xlsx'),
    ('November 2024', 'All Calls by Month/November 2024.xlsx'),
    ('September 2024', 'All Calls by Month/September 2024.xlsx'),
]

# Read and combine all DataFrames
dfs = []

for month, path in month_files:
    df = pd.read_excel(path)
    df['Month'] = month

    # moving month column to first column
    cols = df.columns.tolist()
    cols.insert(0, cols.pop(cols.index('Month')))
    df = df[cols]
    
    dfs.append(df)

# Concatenate all into a single DataFrame
all_calls = pd.concat(dfs, ignore_index = True)

In [ ]:
# Special Intake Line Dictionary
intake_line_numbers = {
    "A2J Immigration": 13123478347,
    "A2J Immigration Toll Free": 18882652188,
    "Austin Intake VM": 13124235904,
    "Bankruptcy Helpdesk VM": 13122296344, 
    "CLASP VM": 13124235900,
    "Criminal Records": 13122296071,
    "Education Law Referrals VM": 13123478392,
    "Fair Housing Intake VM": 13124235909, 
    "HIV Intake VM": 13123478309,
    "JEHD": 13122296072,
    "Legal Clinics": 13124235938,
    "OP Appeals Project": 13124312101, 
    "Veterans Rights Project VM": 13123478340,
    "Trafficking Survivors Assistance Project": 13122296073,
    "Nursing Home Ombudsman": 13122296079,
    "Markham Eviction Help Desk": 13122296014,
    "Migrant Legal Assistance Program": 13124312299
}

# CHECK IN WITH KRISH: is "Migrant Legal Assistance Program": 13124312299 a special intake line or a main number line?
    # because as of May 20, this number was called: Farmworker main number calls (312-431-2299)
    # FINAL ANSWER: document it as a special intake line

In [ ]:
# Main Line Number Dictionary
main_line_numbers = {
    
    "Legal Aid main number calls": 13123411070,
    "Transfers to the English main menu": 13125068646,
    "Transfers to the Spanish main menu": 13125068647,
    "Old ADAPT number calls that route to the Legal Aid main menu": 13122296080,
    "Old Benefits Enrollment number calls that route to the Legal Aid main menu": 13123478342
}

In [ ]:
# Creating a set from the previously defined dictionaries for the purpose of quick lookup
special_intake_numbers = set(intake_line_numbers.values())
main_numbers = set(main_line_numbers.values())

# Creating a mapping from number to name for special intake lines and main lines
number_to_special_intake_name = {v: k for k, v in intake_line_numbers.items()}
number_to_main_line_name = {v: k for k, v in main_line_numbers.items()}

# Creating Relevant columns with names of each line based on previous mapping and the dictionary 
all_calls['Special Intake Line Name'] = all_calls['Called number'].map(number_to_special_intake_name)
all_calls['Main Number Name'] = all_calls['Called number'].map(number_to_main_line_name)

# Classifying Line Type
all_calls['Line Type'] = all_calls['Called number'].apply(
    lambda x: (
        "Unknown" if pd.isna(x) else
        "Main Number" if x == main_numbers else
        "Special Intake Lines" if x in special_intake_numbers else
        "Non-Special Intake Lines"
    )
)

In [ ]:
# Adding New Columns to clearly visualize on Tableau

# Convert datetime columns
datetime_columns = ['Start time', 'Answer time', 'Release time', 'Report time']

for col in datetime_columns:
    if col in all_calls.columns:
        all_calls[col] = pd.to_datetime(all_calls[col], errors='coerce')

all_calls['Hour'] = all_calls['Start time'].dt.hour
    # what hour did they call based on start time

all_calls['DayOfWeek'] = all_calls['Start time'].dt.day_name()
    # what day of the week did they call based on start time

all_calls['Day_Type'] = np.where(all_calls['DayOfWeek'].isin(['Saturday', 'Sunday']), 'Weekend', 'Weekday')
    # is it a weekend or a weekday 

all_calls['Month_Day'] = all_calls['Start time'].dt.strftime('%B %-d')
    # need the day of the month and the month itself

all_calls['Month_Year'] = all_calls['Start time'].dt.strftime('%B %Y')
    # need the month and the year

all_calls['Business hours'] = all_calls['Start time'].apply(
    lambda t: 'Business Hours' if pd.to_datetime('08:00:00').time() <= t.time() <= pd.to_datetime('17:00:00').time()
    else 'Outside Business Hours'
)
    # business hours vs non-business hours

hour_labels = [
    f"{(h % 12 or 12)}:00 {'AM' if h < 12 else 'PM'} - {(h % 12 or 12)}:59 {'AM' if h < 12 else 'PM'}"
    for h in range(24)
]
    # creating hour labels for time buckets

all_calls['Time bucket'] = all_calls['Hour'].apply(
    lambda h: f"{(h % 12 or 12)}:00 {'AM' if h < 12 else 'PM'} - {(h % 12 or 12)}:59 {'AM' if h < 12 else 'PM'}"
)
    # adding the time bucket labels

time_bucket_type = CategoricalDtype(categories=hour_labels, ordered=True)
all_calls['Time bucket'] = all_calls['Time bucket'].astype(time_bucket_type)
    # Setting as ordered categorical for Tableau-friendly sorting

In [ ]:
## Defining the conditions for Call Type: Inbound, Outbound, or Internal

conditions = [
    (all_calls['PSTN vendor name'] == 'CallTower') & (df['Direction'] == 'TERMINATING'),
    (all_calls['PSTN vendor name'] == 'CallTower') & (df['Direction'] == 'ORIGINATING'),
    all_calls['PSTN vendor name'].isna()
]
    # identifying conditions

choices = ['Inbound', 'Outbound', 'Internal']
    # identiying choices based on conditions

all_calls['Call_Type'] = np.select(conditions, choices, default='other')
     # Creating call type 

all_calls['Duration'] = pd.to_numeric(all_calls['Duration'], errors='coerce')
    # making sure duration is in seconds and is numeric

all_calls['Duration_Seconds'] = all_calls['Duration']
all_calls['Duration_Minutes'] = all_calls['Duration'] / 60
    # putting duration in seconds and minutes

all_calls['Date'] = all_calls['Start time'].dt.date
    # need the date

start_date = pd.to_datetime('2024-04-07', utc = True)
end_date = pd.to_datetime('2025-03-15', utc = True)

all_calls = all_calls[(all_calls['Start time'] >= start_date) & (all_calls['Start time'] <= end_date)]

For all inbound calls: To classify Direct vs Transfer, we need to look at two main aspects of the call:

1. Call Type — Is it an "Inbound" call?

2. Call Flow — Did it go directly to a special intake line, or did it first hit a main menu number and then transfer?

In [ ]:
## Defining Direct vs Transfer Calls

all_calls = all_calls.sort_values(by=['Correlation ID', 'Start time'])
    # Sort the dataset to ensure call legs are processed in sequence

all_calls['Inbound_Type'] = None
all_calls['Is_Direct_Inbound'] = False
all_calls['Is_Transfer_Inbound'] = False

# Group by Correlation ID to process each call journey
for cid, group in all_calls.groupby('Correlation ID'):
    group_sorted = group.sort_values(by='Start time')
    
    # Filter to only inbound legs
    inbound_legs = group_sorted[group_sorted['Call_Type'] == 'Inbound']
    
    if inbound_legs.empty:
        continue  # Skip if no inbound legs in this journey
    
    called_numbers = inbound_legs['Called number'].tolist()
    first_called = called_numbers[0]
    
    # Case 1: Direct Inbound - first call went directly to a special intake line
    if first_called in special_intake_numbers:
        all_calls.loc[inbound_legs.index, 'Inbound_Type'] = 'Direct Inbound'
        all_calls.loc[inbound_legs.index, 'Is_Direct_Inbound'] = True
    
    # Case 2: Transfer Inbound - first inbound call is to main line, later call to special intake line
    elif (
        first_called in main_numbers and
        any(inbound_legs['Called number'].isin(special_intake_numbers))
    ):
        # Tag only the FIRST special intake leg
        transfer_leg = inbound_legs[inbound_legs['Called number'].isin(special_intake_numbers)].iloc[0]
        all_calls.loc[transfer_leg.name, 'Inbound_Type'] = 'Transfer Inbound'
        all_calls.loc[transfer_leg.name, 'Is_Transfer_Inbound'] = True


    # Assigning 'Other Inbound' for inbound calls that don't match or fit either category
    else:
        all_calls.loc[inbound_legs.index, 'Inbound_Type'] = 'Other Inbound'

# When a call is transferred to a special intake line, the Correlation ID stays the same, 
# but i may see multiple call legs to the special intake line
# so the purpose of "only tagging the first special intake call leg in a transfer" is 
# to avoid double-counting or overstating the presence of a transfer in the data

In [ ]:
# Creating a Unified Line Category Column for easier visualization in Tableau 

def categorize_line(number):
    if pd.isna(number):
        return "Unknown"
    elif number in special_intake_numbers:
        return "Intake Line"
    elif number in main_numbers:
        return "Main Menu Number"
    else:
        return "Other Line"

all_calls['Line Category'] = all_calls['Called number'].apply(categorize_line)

In [ ]:
# -- Call Leg Tracking --
# Creating a leg group key using only stable identifiers ---
leg_key_cols = [
    'Correlation ID',
    'Date',
    'Start time',
    'Called number',
    'Special Intake Line Name',
    'PSTN vendor name'
]
all_calls['Leg Group Key'] = all_calls[leg_key_cols].astype(str).agg('-'.join, axis=1)

# Assigning leg numbers based on unique leg groups 
journey_groups = all_calls.groupby(['Correlation ID', 'Date'])

all_calls['Leg Number'] = journey_groups['Leg Group Key'].transform(lambda x: pd.factorize(x)[0] + 1)
all_calls['Total Legs'] = journey_groups['Leg Group Key'].transform('nunique')

In [ ]:
print(all_calls.columns)
print(all_calls.shape)

In [ ]:
columns_to_keep = ['Month','Correlation ID', 'Direction', 'Duration_Seconds', 'Duration_Minutes',
                   'Duration', 'Called number', 'Hour','DayOfWeek', 'Date',
                   'PSTN vendor Org ID', 'PSTN vendor name',
                   'Month', 'Start time', 'Answer time', 'Call type',
                   'Call outcome','Special Intake Line Name', 'Main Number Name', 'Line Type', 'Hour',
                   'DayOfWeek', 'Day_Type', 'Month_Day', 'Month_Year', 'Business hours',
                   'Time bucket', 'Call_Type', 'Duration_Seconds', 'Duration_Minutes',
                   'Date', 'Inbound_Type', 'Is_Direct_Inbound', 'Is_Transfer_Inbound',
                   'Leg Group Key', 'Leg Number', 'Total Legs'
                   ]

all_calls_filtered = all_calls[columns_to_keep]

df = pd.DataFrame(all_calls_filtered)

df.to_csv('all_calls_filtered_final.csv', index=False)

In [ ]:
# ## not sure if i need this but

#  Function to flag the first special intake leg after a main line inbound
# def tag_transfer_inbound(group):
#     if group['Is_Inbound'].any() and group['Is_Main_Line'].any():
#         # Find the first special intake leg (if any)
#         special_intake_legs = group[group['Is_Special_Intake']]
#         if not special_intake_legs.empty:
#             first_leg_index = special_intake_legs.index[0]
#             group.at[first_leg_index, 'Transfer_Inbound'] = True
#     return group

# df_tagged = grouped.apply(tag_transfer_inbound)

# def classify_inbound_call_legs(all_calls):
#     all_calls = all_calls.copy()
#     all_calls['Inbound_Type'] = 'Other Inbound'  # Default category
#     all_calls['Is_Direct_Inbound'] = False
#     all_calls['Is_Transfer_Inbound'] = False

#     # Group by Call ID
#     for call_id, group in all_calls.groupby('Call ID'):
#         # Sort by leg start time
#         group_sorted = group.sort_values('Start Time')

#         # Check for direct inbound from main line
#         first_leg = group_sorted.iloc[0]
#         if first_leg['From Number'] in main_numbers and first_leg['Direction'] == 'inbound':
#             all_calls.loc[first_leg.name, 'Inbound_Type'] = 'Direct Inbound'
#             all_calls.loc[first_leg.name, 'Is_Direct_Inbound'] = True

#         # Look for first transfer from special intake line
#         special_transfer_legs = group_sorted[
#             (group_sorted['From Number'].isin(special_intake_numbers)) &
#             (group_sorted['Direction'] == 'inbound')
#         ]
#         if not special_transfer_legs.empty:
#             first_transfer_leg = special_transfer_legs.iloc[0]
#             all_calls.loc[first_transfer_leg.name, 'Inbound_Type'] = 'Transfer Inbound'
#             all_calls.loc[first_transfer_leg.name, 'Is_Transfer_Inbound'] = True

#     return all_calls

# ### define direct vs transfer

# def classify_direct_transfer(group):
#     group_sorted = group.sort_values(by='Start time')
#     first_called = group_sorted.iloc[0]['Called number']
    
#     if first_called in special_intake_numbers:
#         return pd.Series(['Direct'] * len(group), index=group.index)
#     elif first_called in main_numbers and any(group_sorted['Called number'].isin(special_intake_numbers)):
#         return pd.Series(['Transfer'] * len(group), index=group.index)
#     else:
#         return pd.Series(['Other'] * len(group), index=group.index)

# # Apply classification
# all_calls['Direct_v_Transfer'] = all_calls.groupby('Correlation ID', group_keys=False).apply(classify_direct_transfer)


# # Define the known line groups
# special_intake_lines = ['line_1', 'line_2', 'line_3']  # fill in real numbers
# main_menu_numbers = ['main_1', 'main_2', 'main_3']     # fill in real numbers

# # Filter for inbound calls only
# inbound_df = df[df['call_direction'] == 'Inbound'].copy()

# # Convert 'start_time' to datetime if not already
# inbound_df['start_time'] = pd.to_datetime(inbound_df['start_time'])

# # Sort by correlation_id and start_time
# inbound_df.sort_values(by=['correlation_id', 'start_time'], inplace=True)

# # Create a column to store call type
# inbound_df['call_type'] = 'Unknown'

# # Group by unique call
# grouped = inbound_df.groupby('correlation_id')

# # Function to classify each call
# def classify_call(group):
#     first_leg = group.iloc[0]
#     all_called_numbers = group['called_number'].tolist()
    
#     if first_leg['called_number'] in special_intake_lines:
#         return ['Direct'] * len(group)
#     elif first_leg['called_number'] in main_menu_numbers:
#         if any(num in special_intake_lines for num in all_called_numbers[1:]):
#             return ['Transfer'] * len(group)
#     return ['Other'] * len(group)

# # Apply the classification
# inbound_df['call_type'] = grouped.apply(classify_call).explode().values

# # Final output
# inbound_df.head()

# ## adds boolean flag 

# # Step 1: Create empty columns to fill later
# df['Inbound_Type'] = 'Other'
# df['Is_Direct_Inbound'] = False
# df['Is_Transfer_Inbound'] = False

# # Step 2: Sort data to ensure legs are grouped and in order
# df = df.sort_values(by=['Correlation ID', 'Start Time'])

# # Step 3: Function to classify inbound calls
# def classify_inbound_calls(group):
#     """
#     For each Correlation ID group, classify the type of inbound call.
#     """
#     # Check if the group has any inbound calls
#     inbound_calls = group[group['Call_Type'] == 'Inbound']

#     # No inbound? Skip
#     if inbound_calls.empty:
#         return group

#     # Check if any inbound call goes directly to special intake
#     direct = inbound_calls[inbound_calls['Called number'].isin(special_intake_numbers)]
#     if not direct.empty:
#         group.loc[direct.index, 'Inbound_Type'] = 'Direct Inbound'
#         group.loc[direct.index, 'Is_Direct_Inbound'] = True
#         return group

#     # If no direct, check for transfer pattern: first to main, then to special
#     ordered_inbound = inbound_calls.sort_values(by='Start Time')
#     called_sequence = ordered_inbound['Called number'].tolist()

#     first_called = called_sequence[0] if called_sequence else None
#     later_called = called_sequence[1:] if len(called_sequence) > 1 else []

#     # First to main line, later to special intake
#     if (first_called in main_numbers and 
#         any(number in special_intake_numbers for number in later_called)):
#         # Find all inbound calls in group that went to special intake
#         transfer = inbound_calls[inbound_calls['Called number'].isin(special_intake_numbers)]
#         group.loc[transfer.index, 'Inbound_Type'] = 'Transfer Inbound'
#         group.loc[transfer.index, 'Is_Transfer_Inbound'] = True

#     return group

# # Step 4: Apply function to each Correlation ID group
# df = df.groupby('Correlation ID', group_keys=False).apply(classify_inbound_calls)


In [ ]:
# # Filter to inbound calls only
# inbound_calls = all_calls[all_calls['Direction'] == 'Inbound'].copy()

# # --- Sort data to ensure call legs are in correct sequence ---
# # Each call journey is tracked by Correlation ID and timestamp
# inbound_calls.sort_values(by=['Correlation ID', 'Start Time'], inplace=True)

# # --- Identify the first leg of each call journey ---
# # This is used to check what number was originally called
# inbound_calls['Is First Leg'] = inbound_calls['Correlation ID'] != inbound_calls['Correlation ID'].shift(1)
# first_legs = inbound_calls[inbound_calls['Is First Leg']].copy()

# # --- Create a dictionary to map Correlation ID to its initial called number ---
# first_leg_called_number = dict(zip(first_legs['Correlation ID'], first_legs['Called number']))

# # --- Apply classification logic ---
# def classify_call(row):
#     cid = row['Correlation ID']
#     first_called_number = first_leg_called_number.get(cid)

#     if first_called_number in special_intake_numbers:
#         return 'Direct Inbound'
#     elif first_called_number in main_numbers and row['Called number'] in special_intake_numbers:
#         return 'Transfer Inbound'
#     else:
#         return 'Other Inbound'

# # Only apply to inbound_calls (since we've already filtered by direction)
# inbound_calls['Inbound Type'] = inbound_calls.apply(classify_call, axis=1)

# # Now you have a column 'Inbound Type' that shows whether it's:
# # - Direct Inbound (started on a special intake line)
# # - Transfer Inbound (started on a main line, then went to a special intake line)
# # - Other Inbound (doesn't match the logic, possibly non-transfer or unrelated)


In [ ]:
# # Adding many extra columns and time/date features for ease of Tableau visualization
# df['Hour'] = df['Start time'].dt.hour
# df['Start time (Hour/Min)'] = df['Start time'].dt.strftime('%I:%M %p')
# df['Month_Day'] = df['Start time'].dt.strftime('%B %-d')
# df['Day of week'] = df['Start time'].dt.day_name()
# df['Month_Year'] = df['Start time'].dt.strftime('%B %Y')
# df['Date only'] = df['Start time'].dt.date

# # Adding Weekend column
# df['Weekend'] = df['Day of week'].apply(
#     lambda day: 'Weekend' if day in ['Saturday', 'Sunday'] else 'Weekday'
# )

# df['Business hours'] = df['Start time'].apply(
#     lambda t: 'Business Hours' if pd.to_datetime('08:00:00').time() <= t.time() <= pd.to_datetime('17:00:00').time()
#     else 'Outside Business Hours'
# )


# # Adding time bucket labels
# hour_labels = [
#     f"{(h % 12 or 12)}:00 {'AM' if h < 12 else 'PM'} - {(h % 12 or 12)}:59 {'AM' if h < 12 else 'PM'}"
#     for h in range(24)
# ]

# df['Time bucket'] = df['Hour'].apply(
#     lambda h: f"{(h % 12 or 12)}:00 {'AM' if h < 12 else 'PM'} - {(h % 12 or 12)}:59 {'AM' if h < 12 else 'PM'}"
# )

# # Setting as ordered categorical for Tableau-friendly sorting
# time_bucket_type = CategoricalDtype(categories=hour_labels, ordered=True)
# df['Time bucket'] = df['Time bucket'].astype(time_bucket_type)

# # Function to classify call types
# def determine_call_type(row):
#     if pd.isna(row['PSTN vendor name']):
#         return 'Internal Transfer'
#     elif row['PSTN vendor name'] == 'CallTower' and row['Direction'] == 'TERMINATING':
#         return 'Inbound'
#     elif row['PSTN vendor name'] == 'CallTower' and row['Direction'] == 'ORIGINATING':
#         return 'Outbound'
#     else:
#         return 'Unknown'

# df['Prelim Call Type'] = df.apply(determine_call_type, axis=1)

# # Assigning call type and deduplicating internal transfers
# df['Call Type'] = df['Prelim Call Type']  

# internal_transfer_mask = df['Prelim Call Type'] == 'Internal Transfer'

# # marking only the first instance of internal transfer call
# df.loc[internal_transfer_mask, 'Transfer Group'] = df[internal_transfer_mask].groupby(
#     ['Correlation ID', 'Start time']
# ).cumcount()

# # Then marking the duplicates
# df.loc[(df['Prelim Call Type'] == 'Internal Transfer') & (df['Transfer Group'] > 0), 'Call Type'] = 'Duplicate Transfer'

# df.drop(columns=['Prelim Call Type', 'Transfer Group'], inplace=True)

# # -- Repeat Callers -- 
# # Identifing repeat callers (taking into account Correlation ID across multiple days) ---
# call_days = df.groupby('Correlation ID')['Date only'].nunique()
# repeat_flags = call_days[call_days > 1].index
# df['Is Repeat Caller'] = df['Correlation ID'].isin(repeat_flags)

# # -- Call Leg Tracking --
# # Creating a leg group key using only stable identifiers ---
# leg_key_cols = [
#     'Correlation ID',
#     'Date only',
#     'Start time',
#     'Called number',
#     'Intake Line Name',
#     'PSTN vendor name'
# ]
# df['Leg Group Key'] = df[leg_key_cols].astype(str).agg('-'.join, axis=1)

# # Assigning leg numbers based on unique leg groups 
# journey_groups = df.groupby(['Correlation ID', 'Date only'])

# df['Leg Number'] = journey_groups['Leg Group Key'].transform(lambda x: pd.factorize(x)[0] + 1)
# df['Total Legs'] = journey_groups['Leg Group Key'].transform('nunique')